In [1]:
"""
Enfoque alternativo: clustering GLOBAL de todos los embeddings de un modelo
juntos (no por palabra) -- se ajusta UMAP sobre todo el vocabulario a la vez,
se corre DBSCAN, se descartan clusters demasiado grandes (el "blob" genérico)
o de ruido, y se mira en cuántos clusters "decentes" distintos cae cada
palabra.

Se calibra con el MISMO criterio y las MISMAS palabras de control
(monosémicas + polisémicas) que clusterizar_sentidos.py, para poder comparar
directamente contra el resultado del enfoque por palabra (Youden J≈0.22) --
no se adopta este método solo porque un caso ("cambio") se vio bien; se
valida con el mismo rigor.

Requiere: umap-learn, scikit-learn, pandas, numpy
Ejecuta: pip install umap-learn scikit-learn pandas numpy --break-system-packages
"""

import os
import numpy as np
import pandas as pd
import umap
from sklearn.cluster import DBSCAN

C:\Users\msubr\Anaconda3\envs\gpu_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
CARPETA_EMBEDDINGS = "embeddings"
MODELOS = ["mex_state", "mex_large", "beto", "bertin", "robertuito"]

UMAP_N_COMPONENTS_CLUSTERING = 10   # para clustering -- NO usar 2D aquí, eso es solo para graficar
DBSCAN_EPS = 0.1
DBSCAN_MIN_SAMPLES = 8
TAMANO_MAX_DECENTE = 1000  # clusters con más miembros que esto se consideran "el blob genérico", se descartan
MUESTRA_MAXIMA = 100000      # None = usar todas las ocurrencias; poner un entero para submuestrear si el corpus es muy grande

PALABRAS_CALIBRACION_POLISEMICAS = {"banco", "gato", "planta", "sierra", "cola", "papa", "vela", "muñeca"}
PALABRAS_CALIBRACION_MONOSEMICAS = {"bicicleta", "computadora", "cumpleaños", "desayuno",
                                      "aeropuerto", "estudiante", "diciembre", "elefante"}

In [3]:
def cargar_embeddings(nombre_modelo, carpeta=CARPETA_EMBEDDINGS):
    ruta = os.path.join(carpeta, f"{nombre_modelo}_embeddings.npz")
    data = np.load(ruta, allow_pickle=True)
    emb = data["embeddings"]
    words = data["words"]
    id_tweets = data["id_tweets"]
    return emb, words, id_tweets


def clustering_global(nombre_modelo, muestra_maxima=MUESTRA_MAXIMA):
    """Corre UMAP+DBSCAN sobre TODOS los embeddings de un modelo a la vez.
    Devuelve un DataFrame (palabra, id_tweet, cluster) y el array de clusters
    "decentes" (ni ruido -1, ni el blob >TAMANO_MAX_DECENTE)."""
    emb, words, id_tweets = cargar_embeddings(nombre_modelo)
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)

    if muestra_maxima is not None and len(emb) > muestra_maxima:
        # muestreo ALEATORIO -- nunca los primeros N (sesga hacia id_tweet bajo)
        rng = np.random.default_rng(42)
        idx = rng.choice(len(emb), size=muestra_maxima, replace=False)
        emb, words, id_tweets = emb[idx], words[idx], id_tweets[idx]

    print(f"[{nombre_modelo}] ajustando UMAP sobre {len(emb)} ocurrencias...")
    reducer = umap.UMAP(metric="cosine", n_components=UMAP_N_COMPONENTS_CLUSTERING, n_neighbors=15)
    u = reducer.fit_transform(emb)

    print(f"[{nombre_modelo}] corriendo DBSCAN...")
    clusterer = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES)
    labels = clusterer.fit_predict(u)

    df = pd.DataFrame({"palabra": words, "id_tweet": id_tweets, "cluster": labels})

    tam_cluster = df["cluster"].value_counts()
    clusters_decentes = set(tam_cluster[(tam_cluster.index != -1) & (tam_cluster <= TAMANO_MAX_DECENTE)].index)
    print(f"[{nombre_modelo}] clusters totales: {tam_cluster.shape[0]}, "
          f"decentes (no ruido, tamaño<={TAMANO_MAX_DECENTE}): {len(clusters_decentes)}")

    return df, clusters_decentes


def contar_clusters_decentes_por_palabra(df, clusters_decentes):
    """Para cada palabra, cuántos clusters DECENTES distintos ocupa."""
    def _contar(grupo):
        return len(set(grupo) & clusters_decentes)
    return df.groupby("palabra")["cluster"].apply(_contar).rename("num_clusters_decentes")


def calibrar_global(nombre_modelo, muestra_maxima=MUESTRA_MAXIMA):
    """Corre el clustering global completo, y evalúa específicamente las
    palabras de control (monosémicas + polisémicas) -- mismo criterio de
    evaluación que clusterizar_sentidos.py, para comparar directamente."""
    df, clusters_decentes = clustering_global(nombre_modelo, muestra_maxima)
    conteo = contar_clusters_decentes_por_palabra(df, clusters_decentes)

    palabras_calibracion = PALABRAS_CALIBRACION_POLISEMICAS | PALABRAS_CALIBRACION_MONOSEMICAS
    filas = []
    for palabra in palabras_calibracion:
        if palabra not in conteo.index:
            print(f"  [{nombre_modelo}] '{palabra}' no aparece en la muestra usada -- sáltala o sube MUESTRA_MAXIMA")
            continue
        tipo_esperado = "polisémica" if palabra in PALABRAS_CALIBRACION_POLISEMICAS else "monosémica"
        filas.append({
            "modelo": nombre_modelo, "palabra": palabra, "tipo_esperado": tipo_esperado,
            "num_clusters_decentes": conteo[palabra],
        })
    return pd.DataFrame(filas), df, clusters_decentes

In [4]:
if __name__ == "__main__":
    resultados = []
    for nombre_modelo in MODELOS:
        cal_df, _, _ = calibrar_global(nombre_modelo)
        resultados.append(cal_df)

    calibracion_global_df = pd.concat(resultados, ignore_index=True)
    calibracion_global_df.to_csv("calibracion_global.csv", index=False)
    print("\n=== Resultado de calibración (clustering global) ===")
    print(calibracion_global_df.sort_values(["tipo_esperado", "palabra", "modelo"]).to_string(index=False))

    # Sensibilidad/especificidad con el criterio "num_clusters_decentes >= 2 => polisémica",
    # el análogo directo a lo que hicimos con silhouette en el enfoque por palabra.
    es_real = (calibracion_global_df["tipo_esperado"] == "polisémica").values
    predicho_polisemica = (calibracion_global_df["num_clusters_decentes"] >= 2).values
    sensibilidad = (predicho_polisemica & es_real).sum() / es_real.sum()
    especificidad = (~predicho_polisemica & ~es_real).sum() / (~es_real).sum()
    print(f"\nSensibilidad (regla num_clusters_decentes>=2): {sensibilidad:.3f}")
    print(f"Especificidad: {especificidad:.3f}")
    print(f"Youden J: {sensibilidad + especificidad - 1:.3f}")
    print("\nCompara este J contra el del enfoque por palabra (0.215) antes de decidir cuál usar.")

[mex_state] ajustando UMAP sobre 100000 ocurrencias...
[mex_state] corriendo DBSCAN...
[mex_state] clusters totales: 2660, decentes (no ruido, tamaño<=1000): 2658
  [mex_state] 'cumpleaños' no aparece en la muestra usada -- sáltala o sube MUESTRA_MAXIMA
  [mex_state] 'muñeca' no aparece en la muestra usada -- sáltala o sube MUESTRA_MAXIMA
[mex_large] ajustando UMAP sobre 100000 ocurrencias...
[mex_large] corriendo DBSCAN...
[mex_large] clusters totales: 3261, decentes (no ruido, tamaño<=1000): 3259
  [mex_large] 'cumpleaños' no aparece en la muestra usada -- sáltala o sube MUESTRA_MAXIMA
  [mex_large] 'muñeca' no aparece en la muestra usada -- sáltala o sube MUESTRA_MAXIMA
[beto] ajustando UMAP sobre 100000 ocurrencias...
[beto] corriendo DBSCAN...
[beto] clusters totales: 1918, decentes (no ruido, tamaño<=1000): 1915
  [beto] 'cumpleaños' no aparece en la muestra usada -- sáltala o sube MUESTRA_MAXIMA
  [beto] 'muñeca' no aparece en la muestra usada -- sáltala o sube MUESTRA_MAXIMA
[b

In [32]:
# %% [markdown]
# ### Inspección caso por caso (opcional)

# %%
# Ejemplo — ajusta modelo, palabra y parámetros elegidos:
# inspeccionar_ocurrencias(
#     "diciembre",
#     embeddings_calib_por_modelo["beto"],
#     tweets_calib_por_modelo["beto"],
#     min_cluster_size=5, min_samples=HDBSCAN_MIN_SAMPLES, n_componentes_quitar=1,
# )

# %% [markdown]
# ## 4. Orquestación — correr los 5 modelos sobre el corpus completo
#
# `N_COMPONENTES_QUITAR_FINAL = 0`: el whitening no demostró distinguir
# separaciones reales de espurias (sección 3), así que no se usa una
# corrección sin evidencia de que funcione — se mantiene el pipeline simple.
# `presenta_multiples_clusters` describe estructura de embeddings observada,
# no polisemia validada (ver limitación en el encabezado del notebook).
# Guarda incrementalmente (append + flush por palabra) en un CSV nuevo, y es
# reanudable si el proceso se cae a mitad.

# %%
N_COMPONENTES_QUITAR_FINAL = 0    # sin evidencia de que el whitening ayude (sección 3) -- se deja en 0
RESULTADOS_CSV = "resultados_clustering_v2.csv"
CAMPOS_RESULTADOS = ["word_normalized", "modelo", "n_ocurrencias", "min_cluster_size_usado",
                      "n_componentes_quitar", "k_clusters", "pct_ruido", "silhouette",
                      "presenta_multiples_clusters", "excluida_stopword"]
UMBRAL_RUIDO_NO_DETERMINADO = 50  # % de ruido a partir del cual el cluster no es confiable / no se reporta


def cargar_procesados(ruta=RESULTADOS_CSV):
    """Devuelve el set de (modelo, word_normalized) ya procesados, leyendo el
    CSV si existe. Tolera una última línea corrupta (ej. si el proceso se cayó
    a mitad de una escritura)."""
    if not os.path.exists(ruta):
        return set()
    try:
        df_previo = pd.read_csv(ruta, on_bad_lines="skip")
    except TypeError:  # pandas viejo sin on_bad_lines
        df_previo = pd.read_csv(ruta, error_bad_lines=False)
    return set(zip(df_previo["modelo"], df_previo["word_normalized"]))


def clusterizar_todas_las_palabras_resumible(nombre_modelo, min_cluster_size=None, min_samples=None,
                                               n_componentes_quitar=N_COMPONENTES_QUITAR_FINAL, ruta=RESULTADOS_CSV):
    """Igual que clusterizar_palabra, pero recorre TODAS las palabras del
    modelo y guarda el resultado de CADA una apenas termina (append + flush a
    `ruta`), saltándose las que ya estén en el archivo — así, si el proceso se
    cae a mitad, vuelves a correr esta misma función y retoma donde se quedó."""
    emb, w, id_tweets = cargar_embeddings(nombre_modelo)
    emb = emb / np.linalg.norm(emb, axis=1, keepdims=True)
    palabras_arr = np.array(w)
    palabras_unicas = np.unique(palabras_arr)

    procesados = cargar_procesados(ruta)
    escribir_header = not os.path.exists(ruta)
    ya_hechas = sum(1 for p in palabras_unicas if (nombre_modelo, p) in procesados)
    print(f"--- {nombre_modelo}: {ya_hechas}/{len(palabras_unicas)} palabras ya procesadas, retomando ---")

    with open(ruta, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CAMPOS_RESULTADOS)
        if escribir_header:
            writer.writeheader()
            f.flush()

        for i, palabra in enumerate(palabras_unicas):
            if (nombre_modelo, palabra) in procesados:
                continue

            if palabra in STOPWORDS_ES:
                writer.writerow({
                    "word_normalized": palabra, "modelo": nombre_modelo,
                    "n_ocurrencias": None, "min_cluster_size_usado": None,
                    "n_componentes_quitar": None, "k_clusters": None, "pct_ruido": None,
                    "silhouette": None,
                    "presenta_multiples_clusters": None, "excluida_stopword": True,
                })
                f.flush()
                continue

            idx = np.where(palabras_arr == palabra)[0]
            embs_palabra = emb[idx]
            n = len(embs_palabra)

            if n < MIN_OCURRENCIAS_PARA_CLUSTER:
                k, pct_ruido, mcs_usado, silhouette, presenta_multiples_clusters = None, None, None, None, None
            else:
                mcs = min_cluster_size if min_cluster_size is not None else calcular_min_cluster_size(n)
                ms = min_samples if min_samples is not None else HDBSCAN_MIN_SAMPLES
                labels, silhouette = _hdbscan_labels(embs_palabra, mcs, ms, n_componentes_quitar=n_componentes_quitar)
                k = len(set(labels) - {-1})
                pct_ruido = (labels == -1).mean() * 100
                mcs_usado = mcs
                presenta_multiples_clusters = None if pct_ruido > UMBRAL_RUIDO_NO_DETERMINADO else (k > 1)

            writer.writerow({
                "word_normalized": palabra, "modelo": nombre_modelo,
                "n_ocurrencias": n, "min_cluster_size_usado": mcs_usado,
                "n_componentes_quitar": n_componentes_quitar if n >= MIN_OCURRENCIAS_PARA_CLUSTER else None,
                "k_clusters": k, "pct_ruido": pct_ruido, "silhouette": silhouette,
                "presenta_multiples_clusters": presenta_multiples_clusters, "excluida_stopword": False,
            })
            f.flush()  # persiste en disco apenas termina esta palabra, no al final del loop

            if i % 500 == 0:
                print(f"  [{nombre_modelo}] {i}/{len(palabras_unicas)} palabras procesadas")

    print(f"--- {nombre_modelo}: completado ---")


for nombre_modelo in MODELOS:
    clusterizar_todas_las_palabras_resumible(nombre_modelo)

resultados_df = pd.read_csv(RESULTADOS_CSV)
print(resultados_df.head(20))
print(f"\nTotal de filas (palabra x modelo): {len(resultados_df)}")

--- mex_state: 12434/12434 palabras ya procesadas, retomando ---
--- mex_state: completado ---
--- mex_large: 12434/12434 palabras ya procesadas, retomando ---
--- mex_large: completado ---
--- beto: 12434/12434 palabras ya procesadas, retomando ---
--- beto: completado ---
--- bertin: 12434/12434 palabras ya procesadas, retomando ---
--- bertin: completado ---
--- robertuito: 12434/12434 palabras ya procesadas, retomando ---
--- robertuito: completado ---
   word_normalized     modelo  n_ocurrencias  min_cluster_size_usado  \
0           aaaaah  mex_state              8                     5.0   
1            aaaah  mex_state              7                     NaN   
2            aaaay  mex_state              6                     NaN   
3             aaah  mex_state              7                     NaN   
4            abajo  mex_state             98                     5.0   
5       abandonada  mex_state              9                     5.0   
6      abandonadas  mex_state      